In [ ]:
# Colab setup
import sys
if "google.colab" in sys.modules:
    %pip install -q pyedautils ephem statsmodels plotly kaleido ipywidgets

import plotly.io as pio
pio.renderers.default = "notebook_connected"

# InfluxDB — Time Series Database

[InfluxDB](https://www.influxdata.com/) is an open-source **time series database** specifically optimized for storing, querying, and analyzing time-stamped data. It is widely used in monitoring, IoT, and energy data applications.

There are two major versions:

- **InfluxDB v1.x** — uses InfluxQL (an SQL-like query language) and organizes data into databases and retention policies.
- **InfluxDB v2.x** — introduces Flux as the primary query language and organizes data into buckets and organizations.

### Why use InfluxDB for energy data?

- **High write throughput** — designed to ingest millions of data points per second, which is ideal for sensor and meter data.
- **Built-in downsampling** — aggregate high-resolution data into lower-resolution summaries automatically.
- **Retention policies** — automatically expire old data after a configurable period, keeping storage costs manageable.

```{note}
This is a conceptual and reference chapter. Running the code cells requires access to an actual InfluxDB instance. All connection examples use placeholder credentials.
```

## Installation

Install the `influxdb-toolkit` helper package:

```
pip install influxdb-toolkit
```

Alternatively, you can install the official client libraries directly:

- For **v2**: `pip install influxdb-client`
- For **v1**: `pip install influxdb`

In [ ]:
try:
    from influxdb_toolkit import InfluxDB
    print("influxdb-toolkit is available")
except ImportError:
    print("influxdb-toolkit is not installed. Install with: pip install influxdb-toolkit")

## Connecting to InfluxDB v2

InfluxDB v2 uses **tokens** for authentication and organizes data into **organizations** and **buckets**. The example below shows how to establish a connection using placeholder credentials.

In [ ]:
try:
    client = InfluxDB(
        url="http://localhost:8086",  # placeholder URL
        token="my-token",             # placeholder token
        org="my-org",                 # placeholder organization
        bucket="my-bucket",           # placeholder bucket
        version=2
    )
    print("Connected to InfluxDB v2")
except Exception as e:
    print(f"Connection not available: {e}")

## Connecting to InfluxDB v1

InfluxDB v1 uses **username/password** authentication and organizes data into **databases** with **retention policies**.

In [ ]:
try:
    client_v1 = InfluxDB(
        url="http://localhost:8086",  # placeholder URL
        username="my-user",           # placeholder username
        password="my-password",       # placeholder password
        database="my-database",       # placeholder database
        version=1
    )
    print("Connected to InfluxDB v1")
except Exception as e:
    print(f"Connection not available: {e}")

## Querying Time Series Data

Once connected, you can query time series data as a pandas DataFrame. The example below retrieves temperature measurements from the last 30 days.

In [ ]:
try:
    df = client.get_timeseries(
        measurement="temperature",
        start="-30d",
        fields=["value"]
    )
    print(df.head())
except Exception as e:
    print(f"Query not available (no database connection): {e}")

## Data Schema

InfluxDB organizes data using a specific schema model. Understanding these concepts is essential for designing efficient time series storage.

| Concept         | Description                                                                 |
|-----------------|-----------------------------------------------------------------------------|
| **Measurement** | The name of the data series (similar to a table in SQL), e.g. `temperature` |
| **Tags**        | Indexed key-value pairs used for filtering and grouping, e.g. `location=office` |
| **Fields**      | The actual data values (not indexed), e.g. `value=21.5`                     |
| **Timestamp**   | The time associated with each data point (nanosecond precision)             |

A single data point in InfluxDB line protocol looks like this:

```
temperature,location=office,sensor=s1 value=21.5 1609459200000000000
```

Where:
- `temperature` is the **measurement**
- `location=office,sensor=s1` are **tags**
- `value=21.5` is a **field**
- `1609459200000000000` is the **timestamp** (Unix nanoseconds)

```{tip}
Choose tags carefully. Tags are indexed and should be used for metadata you want to filter or group by (e.g. building, floor, sensor ID). Fields hold the actual measured values and are not indexed.
```

```{tip}
In production, never hard-code credentials in your notebooks. Store them in **environment variables** or a **`.env` file** and load them with `os.environ` or a library like `python-dotenv`.
```

```python
import os

url = os.environ.get("INFLUXDB_URL", "http://localhost:8086")
token = os.environ.get("INFLUXDB_TOKEN")
```